# Alpaca connection test
Run all cells using the repository `.venv` kernel. This notebook loads the repository's `.env.local` (or `env.local`) and performs read-only requests. It never places orders or displays credentials, account identifiers, or balances.

The account check verifies the credentials against `ALPACA_BASE_URL`. The stock check explicitly uses IEX. The final Bitcoin example uses public data and does not prove that your credentials work.

In [1]:
from pathlib import Path
from datetime import datetime, timezone

try:
    from dotenv import dotenv_values
    from alpaca.trading.client import TradingClient
    from alpaca.data.historical import StockHistoricalDataClient, CryptoHistoricalDataClient
    from alpaca.data.requests import StockLatestQuoteRequest, CryptoBarsRequest
    from alpaca.data.enums import DataFeed
    from alpaca.data.timeframe import TimeFrame
    from alpaca.common.exceptions import APIError
    from requests.exceptions import RequestException
except ImportError:
    raise RuntimeError('Select the repository .venv kernel, or install dependencies with %pip install alpaca-py python-dotenv') from None

# Find the repository from either the root or the notebook directory.
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'Simulated Engine' / 'pyproject.toml').is_file() and (p / 'Alpaca').is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError('Open this notebook from within the FMS repository.')
env_path = next((repo_root / name for name in ('.env.local', 'env.local')
                 if (repo_root / name).is_file()), None)
if env_path is None:
    raise FileNotFoundError('Create .env.local in the FMS repository root.')

# Read fresh values on every run; stale kernel environment variables cannot override the file.
settings = dotenv_values(env_path)
required = ('ALPACA_API_KEY_ID', 'ALPACA_API_SECRET_KEY', 'ALPACA_BASE_URL')
missing = [name for name in required if not (settings.get(name) or '').strip()]
if missing:
    raise ValueError('Missing configuration: ' + ', '.join(missing))
api_key = settings['ALPACA_API_KEY_ID'].strip()
secret_key = settings['ALPACA_API_SECRET_KEY'].strip()
base_url = settings['ALPACA_BASE_URL'].strip().rstrip('/')
# Accept a configured /v2 suffix, but only send keys to official Alpaca hosts.
if base_url.endswith('/v2'):
    base_url = base_url[:-3]
if base_url not in ('https://paper-api.alpaca.markets', 'https://api.alpaca.markets'):
    raise ValueError('ALPACA_BASE_URL must be https://paper-api.alpaca.markets or https://api.alpaca.markets (optionally ending in /v2).')
paper = base_url == 'https://paper-api.alpaca.markets'
trading_client = TradingClient(api_key, secret_key, paper=paper)
stock_client = StockHistoricalDataClient(api_key, secret_key)
print(f'Loaded {env_path.name}; account environment: {"paper" if paper else "live"}.')


def check_request(label, request):
    # Do not echo exception bodies, request headers, or credentials into notebook output.
    try:
        result = request()
    except APIError as exc:
        status = exc.status_code
        hints = {
            401: 'Check the key pair and whether it belongs to the configured paper/live environment.',
            403: 'Check account permissions and data-feed access.',
            429: 'Rate limit reached; wait before rerunning.',
        }
        raise RuntimeError(f'{label} failed (HTTP {status}). ' + hints.get(status, 'Check Alpaca service availability and request settings.')) from None
    except RequestException:
        raise RuntimeError(f'{label} failed to connect. Check network, proxy, and TLS settings.') from None
    print(f'{label}: PASS')
    return result

Loaded .env.local; account environment: paper.


## 1. Authenticate against the configured account
A successful response proves the key pair works for the selected paper or live endpoint. Only the account status is shown.

In [2]:
account = check_request('Account authentication', trading_client.get_account)
print(f'Account status: {account.status.value}')

Account authentication: PASS
Account status: ACTIVE


## 2. Request an authenticated stock quote
IEX is explicitly selected instead of the default feed. A quote timestamp may be from a previous session when the market is closed; this check verifies access, not freshness.

Reference: [Alpaca stock historical data client](https://alpaca.markets/sdks/python/api_reference/data/stock/historical.html).

In [3]:
quotes = check_request(
    'Stock market data (IEX)',
    lambda: stock_client.get_stock_latest_quote(
        StockLatestQuoteRequest(symbol_or_symbols=['AAPL'], feed=DataFeed.IEX)
    ),
)
quote = quotes.get('AAPL')
if quote is None:
    raise RuntimeError('The request succeeded but returned no AAPL quote.')
print(f'AAPL | bid {quote.bid_price} | ask {quote.ask_price} | timestamp {quote.timestamp}')

Stock market data (IEX): PASS
AAPL | bid 325.87 | ask 325.99 | timestamp 2026-09-03 13:57:48.794986+00:00


## 3. Original public Bitcoin history example
This request does not require credentials. Dates are explicitly UTC.

In [4]:
crypto_client = CryptoHistoricalDataClient()
request_params = CryptoBarsRequest(
    symbol_or_symbols=['BTC/USD'],
    timeframe=TimeFrame.Day,
    start=datetime(2022, 9, 1, tzinfo=timezone.utc),
    end=datetime(2022, 9, 7, tzinfo=timezone.utc),
)
btc_bars = check_request('Public Bitcoin history', lambda: crypto_client.get_crypto_bars(request_params))
if btc_bars.df.empty:
    raise RuntimeError('The request succeeded but returned no Bitcoin bars.')
btc_bars.df

Public Bitcoin history: PASS


open      high       low     close  \
symbol  timestamp                                                           
BTC/USD 2022-09-01 00:00:00+00:00  20051.81  20205.83  19564.86  20132.97   
        2022-09-02 00:00:00+00:00  20132.50  20444.00  19757.72  19954.16   
        2022-09-03 00:00:00+00:00  19950.63  20054.69  19658.04  19832.06   
        2022-09-04 00:00:00+00:00  19834.87  20030.89  19587.86  20002.38   
        2022-09-05 00:00:00+00:00  19998.77  20058.00  19635.96  19795.12   
        2022-09-06 00:00:00+00:00  19795.12  20180.50  18668.90  18790.39   
        2022-09-07 00:00:00+00:00  18789.40  19462.02  18534.06  19290.53   

                                         volume  trade_count          vwap  
symbol  timestamp                                                           
BTC/USD 2022-09-01 00:00:00+00:00   7529.674053     114052.0  19934.701556  
        2022-09-02 00:00:00+00:00   7392.679014      98745.0  20095.899441  
        2022-09-03 00:00:00+00:00   3077.135497      52729.0  19839.406563  
        2022-09-04 00:00:00+00:00   3712.178165      60722.0  19813.537532  
        2022-09-05 00:00:00+00:00   4817.489036      66396.0  19801.578592  
        2022-09-06 00:00:00+00:00  11753.830278     139147.0  19480.986370  
        2022-09-07 00:00:00+00:00   8092.183326      89704.0  18952.481132